# **Primera práctica de Aprendizaje Automático**
### **Predicción de subscripción a un producto bancario**
Paloma González (*100498978*) y Javier Pastor (*100499039*)

## **0. Pasos previos a la ejecución**

### **Instalación de librerías requeridas para el proyecto:**

In [ ]:
%pip install -r requirements.txt

### **Fijación de la semilla para el proyecto:**
Utilizaremos la semilla `12`, puesto que usaremos `ab = 39`, lo que nos da `xx = a+b = 3+9 = 12`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
SEMILLA = 12 
np.random.seed(SEMILLA)
df = pd.read_pickle('bank_ALL/bank_12.pkl')

## **1. EDA simplificado**

#### **Estructura del dataset y tipos de variables:**
   Mediante el uso de `df.info()` y `df.nunique()`, podemos extraer el número total de instancias y variables que forman el conjunto de datos. Asimismo, estas instrucciones nos permiten saber cuáles de estas variables son numéricas (*int64*, *float64*) y cuáles categóricas/ordinales (*object* o *category*).

In [ ]:
df.info()
df.nunique()

#### **Exploración visual de las relaciones multivariantes:**

Para complementar la extracción numérica y hacernos una idea de cómo se relacionan las variables entre sí (y respecto a nuestra variable objetivo *deposit*), generamos las siguientes representaciones gráficas:
- **Pairplot global:** Un análisis bivariante general que nos muestra si existen patrones claros de separabilidad entre las clases 'yes' y 'no' basándonos en las variables numéricas.
- **Pairplot condicionado (sin contacto previo):** Filtramos el dataset para aquellos clientes que no tuvieron contacto previo (`pdays == -1`) generando la misma gráfica. Esto nos ayuda a inferir de un vistazo si el comportamiento de este subgrupo difiere del resto de la base de clientes.
- **Histograma temporal:** Por último, visualizamos la distribución de la característica *day* (día del mes) dentro de dicho subgrupo para encontrar posibles estacionalidades.


In [ ]:
sns.pairplot(df, hue="deposit", markers=["o", "s"], plot_kws={"alpha": 0.3}, corner=True)
plt.show()

In [ ]:
not_previous_contact = df[df['pdays'] == -1]
sns.pairplot(not_previous_contact, hue="deposit", markers=["o", "s"], plot_kws={"alpha": 0.3}, corner=True)
plt.show()

In [ ]:
not_previous_contact['day'].hist()

#### **Análisis y preproceso de la variable *pdays***

Realizamos un análisis particular sobre la variable *pdays* (días desde el último contacto en una campaña anterior). Observamos que una gran mayoría de los registros adoptan el valor de `-1`, que actúa como un flag indicando que el cliente **no había sido contactado previamente**, más que como una medida temporal real.

Mantener este valor numérico en los modelos podría introducir un sesgo indeseado. Por ello, procedemos al preprocesamiento en esta misma etapa mediante dos transformaciones:
- **`wasContacted`**: Una variable binaria que indica si hubo contacto (1) o no (0).
- **`contactedLabel`**: Una discretización (*bining*) que agrupa a los clientes según tramos temporales (0: sin contacto, 1: <100 días, 2: <200 días, etc.), capturando así la relevancia del tiempo transcurrido de forma más estructurada.


In [ ]:
def pdays_transform(x):
    if x == -1:
        return 0
    else:
        return 1


df["pdays"].map(pdays_transform)
df["wasContacted"] = df["pdays"].map(pdays_transform)

df

In [ ]:
def bin_pdays(x):
    if x == -1:
        return 0
    if x < 100:
        return 1
    elif x < 200:
        return 2
    elif x < 400:
        return 3
    else:
        return 4

df['contactedLabel'] = df['pdays'].map(bin_pdays)
df['contactedLabel'].value_counts(sort=True).plot(kind='bar')

#### **Naturaleza del problema y desbalanceo**

Es fundamental conocer nuestra variable objetivo (*deposit*), la cual determina la suscripción o no al producto. Al ser una variable binaria (*yes* or *no*), el contexto se define como un **problema de clasificación**. Mediante las siguientes sentencias calcularemos su distribución exacta para comprobar el nivel de desbalanceo, un factor crucial para decidir más tarde las estrategias de división (estratificación) y métricas de evaluación.


In [ ]:
print("Distribución de la variable objetivo (deposit):")
print(df['deposit'].value_counts())
print("\nPorcentaje de desbalanceo:")
print(df['deposit'].value_counts(normalize=True) * 100)

#### **Comprobaciones estructurales y de calidad de los datos**

Por último, programamos una revisión sistemática del dataset para identificar elementos críticos para el posterior preprocesamiento y entrenamiento:
- **Valores faltantes:** Comprobación automatizada de las variables que contienen elementos nulos.
- **Columnas Constantes e IDs:** Detección de columnas formadas por un solo valor repetido o, por el contrario, que actúan como meros identificadores únicos para cada fila (ambos casos carecen de valor predictivo).
- **Alta Cardinalidad:** Identificación sistemática de las variables categóricas que sobrepasan un umbral de 10 valores únicos, las cuales requerirán atención al aplicar técnicas de codificación para no disparar la dimensionalidad.


In [ ]:
print("Valores nulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0]) # Solo muestra las que tienen nulos

# Columnas constantes (1 solo valor) o IDs (tantos valores como filas)
constantes = [col for col in df.columns if df[col].nunique() == 1]
ids = [col for col in df.columns if df[col].nunique() == len(df)]
print(f"\nColumnas constantes: {constantes}")
print(f"Columnas tipo ID: {ids}")

# Alta cardinalidad en variables categóricas (>10 valores únicos)
categoricas = df.select_dtypes(include=['object', 'category']).columns
alta_card = [col for col in categoricas if df[col].nunique() > 10]
print(f"\nVariables categóricas con alta cardinalidad (>10): {alta_card}")

## **2. División de datos (Holdout)**

#### **Estrategia de evaluación (*outer* e *inner*) y selección de métricas**

Para garantizar que el modelo no solo memorice los datos, sino que generalice bien ante información nueva, hemos definido el siguiente flujo de evaluación:

1. **Evaluación Outer (Holdout):**
   - Siguiendo las instrucciones, dividimos nuestro conjunto original mediante la técnica Holdout: mantendremos **2/3 (66%)** de los datos para entrenamiento (*train*) y **1/3 (33%)** extraído en un compartimento estanco para la evaluación final (*test*).
   - El conjunto de *test* permanecerá totalmente bloqueado durante toda la fase de experimentación. Únicamente lo utilizaremos al final de los respectivos apartados para comprobar de forma genuina el desempeño de nuestros mejores modelos.
   - **Métrica elegida:** Puesto que nuestra variable objetivo presenta un ligero desbalanceo (~47% 'yes' respecto al 52% 'no'), nos centramos especialmente en observar la métrica combinada **F1-Score**, así como el **Recall (Sensibilidad)** de la clase 'yes', pues el objetivo principal de este caso de negocio es no dejar escapar a ningún cliente interesado en suscribir el depósito. También anotaremos la exactitud global (*Accuracy*).

2. **Evaluación Inner (Validación interna):**
   - Puesto que no podemos usar el dataset de prueba (*test*) para ajustar hiperparámetros (HPO), el entrenamiento de cada algoritmo se realizará mediante una estrategia interna de **Validación cruzada** sobre el conjunto de *train*.
   - Usaremos `cv=5` internamente en nuestros procesos de optimización (*GridSearchCV*), lo que asegura que el modelo es validado múltiples veces en particiones distintas del subconjunto de entrenamiento, dotando de mucha más robustez a nuestra decisión de hiperparámetros.

#### **Reestructuración y División del Dataset**

En la siguiente celda procedemos en tres fases:
1. Aislar los atributos dependientes (y) de los predictores (X), eliminando variables ya procesadas en el EDA como *pdays*.
2. Segregar el dataset en conjuntos *train* y *test* utilizando el tamaño definido `test_size=1/3`. Aplicamos el parámetro `stratify=y` para asegurar que la proporción actual de clientes interesados ('yes' / 'no') se mantenga inalterada en ambos conjuntos resultantes.
3. Declarar toda la estandarización y transformación (*SimpleImputer, OneHotEncoder, StandardScaler*) integrándolo todo dentro de un **ColumnTransformer** y un **Pipeline**. Esto evita el riesgo de fuga de información (*Data Leakage*), ya que los escaladores solo "aprenderán" la distribución del *train* (sin haber visto jamás el *test*).


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Separar variables predictoras (X) de la variable objetivo (y)
# Eliminamos 'deposit' (es lo que predecimos) y 'pdays' (usaremos las nuevas que creaste)
X = df.drop(columns=['deposit', 'pdays']) 
y = df['deposit']

# 2. División Train/Test (2/3 para entrenar, 1/3 para testear)
# Usamos stratify=y para que se mantenga el balance ~52/47 en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=1/3, 
    random_state=SEMILLA,
    stratify=y
)

print(f"Filas para Entrenamiento (Train): {X_train.shape[0]}")
print(f"Filas para Evaluación (Test): {X_test.shape[0]}\n")


# PIPELINES

# 1. Identificamos automáticamente qué columnas son números y cuáles texto
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# 2. Qué hacer con los números: Solo escalarlos (Estandarización)
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# 3. Qué hacer con el texto: Rellenar nulos con el valor más frecuente y hacer One-Hot Encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 4. Juntamos todo en un "Preprocesador" maestro
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocesador configurado con éxito.")

## **3. Métodos básicos: KNN y Árboles**

En esta sección abordaremos por primera vez el modelado predictivo utilizando algoritmos de aprendizaje supervisado de corte clásico. El objetivo principal de este apartado es evaluar enfoques algorítmicos distintos, altamente interpretables, para establecer una sólida línea base (*baseline*) de rendimiento. De esta forma, descubriremos si las características de nuestros clientes permiten trazar un perfil fiable de antemano antes de recurrir a técnicas más costosas matemáticamente.


##### Se trabajarán específicamente dos familias de algoritmos:

- **K-Vecinos más cercanos (KNN):** Este algoritmo asume que instancias demográfica y financieramente similares pertenecerán a la misma clase de interés ("suscribir" o "no suscribir"). Durante su ajuste de hiperparámetros (HPO), prestaremos especial atención no sólo a la cantidad de vecinos de voto (*n_neighbors*), sino a medir directamente el impacto que los distintos métodos de escalado (Standard, MinMax, Robust) provocan sobre un algoritmo basado en la distancia espacial euclídea.

- **Árboles de decisión:** Seguidamente, exploraremos la capacidad predictiva de realizar divisiones de datos secuenciales y condicionales. Su principal valor para este problema de negocio radica en su transparencia. En la fase de optimización, controlaremos métricas que limitan el crecimiento salvaje del árbol (como *max_depth* o *min_samples_split*) con el fin primordial de mitigar de forma exhaustiva el sobreajuste (*overfitting*).

### **3.1. Modelo con KNN (*K-vecinos más cercanos*)**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# 1. Crear el Pipeline completo: Preprocesador + Algoritmo KNN
knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier())
])

# 2. Definir los hiperparámetros que queremos que el ordenador pruebe
# n_neighbors: probamos distintos números impares de "vecinos"
# weights: probamos si es mejor que todos los vecinos voten igual ('uniform') o si los más cercanos pesan más ('distance')
param_grid_knn = {
    'classifier__n_neighbors': [3, 5, 7, 9, 11, 15],
    'classifier__weights': ['uniform', 'distance']
}

# 2. Definir los hiperparámetros (¡Añadimos los escaladores al Grid!)
# OJO a la sintaxis: preprocessor__num__scaler nos permite cambiar el escalador del preprocesador
param_grid_knn = {
    'preprocessor__num__scaler': [StandardScaler(), MinMaxScaler(), RobustScaler()],
    'classifier__n_neighbors': [3, 5, 7, 9, 11, 15],
    'classifier__weights': ['uniform', 'distance']
}

# 3. Configurar la búsqueda (GridSearchCV)
grid_knn = GridSearchCV(
    knn_pipeline, 
    param_grid_knn, 
    cv=5, 
    scoring='accuracy', # Ojo: Asegúrate de usar la misma métrica luego en SVM y Regresión Logística
    n_jobs=-1, 
    verbose=1 
)

# 4. Entrenar
grid_knn.fit(X_train, y_train)

print(f"Mejor configuración encontrada (incluyendo escalador): {grid_knn.best_params_}")


# 6. Evaluar el MEJOR modelo usando los datos de Test (la prueba de fuego)
y_pred_knn = grid_knn.predict(X_test)

# 7. Mostrar resultados y métricas
print("\n--- REPORTE DE CLASIFICACIÓN (KNN en datos de Test) ---")
print(classification_report(y_test, y_pred_knn))

# 8. Dibujar la Matriz de Confusión para verlo visualmente
cm_knn = confusion_matrix(y_test, y_pred_knn)
plt.figure(figsize=(6,4))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predice NO', 'Predice YES'], 
            yticklabels=['Real NO', 'Real YES'])
plt.title('Matriz de Confusión - Mejor Modelo KNN')
plt.show()

Tras someter el algoritmo K-Nearest Neighbors a un ajuste de hiperparámetros y evaluación de preprocesamiento mediante *GridSearchCV*, hemos obtenido las siguientes conclusiones:

#### **Configuración óptima:**
El modelo ha determinado que la mejor configuración requiere aplicar una estandarización previa de los datos usando ***StandardScaler***, utilizar **9 vecinos** (`n_neighbors=9`) y dar más importancia a los vecinos más cercanos frente a los lejanos (`weights='distance'`). 

#### **Rendimiento global:**
El modelo logra una **exactitud (accuracy) del 81%** en el conjunto de Test. Esto significa que acierta 8 de cada 10 predicciones en datos que nunca había visto, lo cual es un buen punto de partida.

#### **Análisis de la clase de interés (*yes*):**
En el contexto de una campaña bancaria, nuestro objetivo principal es detectar correctamente a los clientes que SÍ van a suscribir el depósito para optimizar las llamadas telefónicas. Observando las métricas para la clase ***yes***:
* **Precisión (0.82):** Es bastante alta. Significa que cuando nuestro modelo predice que un cliente va a contratar el depósito, acierta el 82% de las veces. Tenemos pocos *falsos positivos*, lo que implica que el banco no perderá recursos llamando a clientes que realmente no están interesados.
* **Recall / Sensibilidad (0.77):** Somos capaces de identificar al 77% del total de clientes que realmente querían el depósito. Sin embargo, se nos escapa un 23% (*falsos negativos*), que son clientes rentables a los que el banco no llamaría porque el modelo falló al predecirlos.
* **F1-Score (0.79):** Nos muestra un buen equilibrio armónico entre la precisión y el recall para esta clase.

Es un modelo conservador y eficiente. Prefiere asegurar el tiro (alta precisión) antes que intentar captar a todos los clientes a costa de equivocarse un poco más (recall moderado).



### **3.2. Modelo Trees (*Árboles de decisión*)**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# 1. Crear el Pipeline: Preprocesador + Algoritmo de Árbol de Decisión
tree_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=39)) # Usa tu semilla aquí
])

# 2. Definir los hiperparámetros a explorar
param_grid_tree = {
    'classifier__max_depth': [None, 5, 10, 15, 20],
    'classifier__min_samples_split': [2, 5, 10, 20],
    'classifier__criterion': ['gini', 'entropy']
}

# 3. Configurar la búsqueda (GridSearchCV)
grid_tree = GridSearchCV(
    tree_pipeline, 
    param_grid_tree, 
    cv=5, 
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# 4. Entrenamiento de árboles
print("Entrenando múltiples Árboles de Decisión para buscar el mejor...\n")
grid_tree.fit(X_train, y_train)

# 5. Mostrar la mejor configuración
print(f"Mejores hiperparámetros encontrados: {grid_tree.best_params_}")

# 6. Evaluar el MEJOR árbol usando los datos de Test
y_pred_tree = grid_tree.predict(X_test)

# 7. Mostrar resultados y métricas
print("\n--- REPORTE DE CLASIFICACIÓN (TREES en datos de Test) ---")
print(classification_report(y_test, y_pred_tree))

# 8. Dibujar la Matriz de Confusión
cm_tree = confusion_matrix(y_test, y_pred_tree)
plt.figure(figsize=(6,4))
sns.heatmap(cm_tree, annot=True, fmt='d', cmap='Greens', 
            xticklabels=['Predice NO', 'Predice YES'], 
            yticklabels=['Real NO', 'Real YES'])
plt.title('Matriz de Confusión - Mejor Árbol de Decisión')
plt.show()

Tras aplicar *GridSearchCV* para encontrar los hiperparámetros óptimos del Árbol de Decisión, extraemos las siguientes conclusiones:

#### **Configuración óptima:**
El modelo ha decidido que la mejor forma de dividir los datos es usando el criterio **'gini'**. Para evitar el sobreajuste (*overfitting*), ha limitado la profundidad máxima del árbol a **10 niveles** (`max_depth=10`) y exige que un nodo tenga al menos **20 muestras** antes de intentar dividirlo más (`min_samples_split=20`).

#### **Rendimiento global:**
Hemos obtenido una **exactitud (accuracy) del 82%** en el conjunto de Test, mejorando ligeramente el resultado que obtuvimos con el modelo KNN.

#### **Análisis de la clase de interés (*yes*):**
* **Precisión (0.83):** Es excelente. De cada 100 personas que el árbol dice que van a contratar, acierta en 83. El banco no desperdiciará recursos en llamadas inútiles.
* **Recall / Sensibilidad (0.79):** Aquí está la principal fortaleza del árbol. Ha logrado capturar al 79% de los clientes que realmente querían el depósito. Hemos reducido notablemente los *Falsos Negativos*.
* **F1-Score (0.81):** Refleja esta mejora general en el equilibrio armónico entre precisión y recall para la clase positiva.

In [ ]:
from sklearn.tree import plot_tree

# 1. Crear y entrenar un árbol poco profundo específicamente para visualizarlo
arbol_simple = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=3, random_state=39, criterion='gini'))
])
arbol_simple.fit(X_train, y_train)

# 2. Extraer el modelo y los nombres de las variables preprocesadas
modelo_arbol = arbol_simple.named_steps['classifier']
nombres_columnas = arbol_simple.named_steps['preprocessor'].get_feature_names_out()

# 3. Dibujar el árbol
plt.figure(figsize=(20, 10))
plot_tree(modelo_arbol, 
          feature_names=nombres_columnas, 
          class_names=['No Contrata', 'Sí Contrata'], 
          filled=True, 
          rounded=True, 
          fontsize=10)
plt.title("Visualización de las Decisiones del Árbol (Profundidad = 3)")
plt.show()

#### **Interpretación de las decisiones del árbol**

Al visualizar el árbol de decisión con profundidad 3, podemos entender las reglas principales que utiliza el modelo para clasificar a los clientes:

* **Variable más determinante (nodo raíz):** La primera división (arriba del todo) se realiza sobre la variable ***num__duration*** (la duración del último contacto). Esto indica que es el atributo que más información aporta para separar a los clientes que contratan de los que no.
* **Regla de descarte rápido:** Si la duración de la llamada es corta (`num__duration <= 0.055`, valor estandarizado), el modelo se va por la rama izquierda, donde la inmensa mayoría de las muestras (nodos naranjas) acaban en la clasificación **'No Contrata'**.
* **Decisiones secundarias:** Si la llamada es más larga (rama derecha), el árbol evalúa ***cat__contact_unknown*** (si el método de contacto es desconocido). Las combinaciones de llamadas de mayor duración donde el método de contacto sí es conocido llevan a los nodos más puros de color azul, indicando una altísima probabilidad de que el cliente **'Sí Contrata'**.

En resumen: El modelo da prioridad absoluta a la duración de la llamada y al método de contacto para perfilar el éxito de la campaña.

### **Comparativa entre KNN y Árboles**

Comparando ambos modelos básicos, **el árbol de decisión es el claro ganador para nuestro problema de negocio**. 

Aunque el árbol de decisión tiene una Precisión ligeramente superior (0.83 vs 0.82) a la hora de no equivocarse al predecir un "Sí", la principal ventaja radica en que tiene un **Recall superior (0.79 vs 0.77)**. En la práctica, esto significa que utilizando el árbol de decisión el banco logrará llamar y captar a un 2% más de clientes reales que estaban dispuestos a contratar el depósito frente al KNN, generando más beneficios para la entidad sin penalizar la tasa de llamadas fallidas.

## **4. Métodos avanzados: modelos lineales y SVMs**

En esta sección, se profundiza en el análisis predictivo mediante el uso de algoritmos de aprendizaje supervisado más avanzados y con diferentes fundamentos matemáticos que los anteriores. El objetivo es contrastar si la aplicación de modelos con fronteras de decisión lineales y técnicas de soporte vectorial permite mejorar la capacidad de generalización obtenida con los métodos básicos.

##### Se trabajarán específicamente dos familias de algoritmos:

- Modelos Lineales (regresión logística): Se evaluará el desempeño del modelo base y se aplicará regularización L1 (Lasso). Esta técnica es de especial interés en este problema bancario, ya que permite realizar una selección de atributos automática al penalizar los coeficientes de las variables menos informativas, facilitando así la interpretación del modelo.

- Máquinas de Vector de Soporte (SVM): Se explorará el uso de SVM, un método potente para encontrar el hiperplano óptimo de separación entre clientes que suscriben el depósito y los que no. Se prestará especial atención al ajuste del parámetro de regularización C y a la elección del kernel para capturar posibles relaciones no lineales en los datos demográficos y financieros.

#### **Preparación de modelos con hiperparámetros por omisión**

Primero, evaluamos los modelos base para tener un punto de referencia y medir los tiempos de entrenamiento.

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Definimos los modelos con sus parámetros por defecto
# Para regularización L1 en LogisticRegression, necesitamos el solver 'liblinear' o 'saga'
models_default = {
    "Logistic Regression (L2)": LogisticRegression(max_iter=1000, random_state=SEMILLA),
    "Logistic Regression (L1)": LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, random_state=SEMILLA),
    "SVM (RBF)": SVC(kernel='rbf', random_state=SEMILLA)
}

results_default = {}

print("Evaluación con hiperparámetros por omisión:")
for name, model in models_default.items():
    # Creamos un pipeline que use el preprocesador que definiste en apartados anteriores
    # Asumo que tu objeto de preprocesamiento se llama 'preprocessor'
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('clf', model)])
    
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Evaluación (usando validación interna o el set de entrenamiento para esta fase inicial)
    # El enunciado pide evaluación inner (puedes usar cross_val_score aquí)
    preds = pipeline.predict(X_train) 
    
    results_default[name] = {
        "Time (s)": train_time,
        "Accuracy (Train)": pipeline.score(X_train, y_train)
    }
    print(f"{name} entrenado en {train_time:.4f} segundos.")

#### **Ajuste de hiperparámetros (HPO)**

Ahora buscamos los mejores valores para cada modelo. Para modelos lineales, solemos ajustar C (fuerza de regularización). Para SVM, ajustamos C y gamma.

In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. HPO para Regresión Logística
param_grid_lr = {
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l1', 'l2'],
    'clf__solver': ['liblinear'] # 'liblinear' soporta l1 y l2
}

grid_lr = GridSearchCV(Pipeline([('preprocessor', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=SEMILLA))]), 
                       param_grid_lr, cv=5, scoring='f1_weighted', n_jobs=-1)

start_lr = time.time()
grid_lr.fit(X_train, y_train)
time_lr_hpo = time.time() - start_lr

# 2. HPO para SVM
param_grid_svm = {
    'clf__C': [0.1, 1, 10],
    'clf__gamma': ['scale', 0.01, 0.1],
    'clf__kernel': ['rbf', 'linear']
}

grid_svm = GridSearchCV(Pipeline([('preprocessor', preprocessor), ('clf', SVC(random_state=SEMILLA))]), 
                        param_grid_svm, cv=5, scoring='f1_weighted', n_jobs=-1)

start_svm = time.time()
grid_svm.fit(X_train, y_train)
time_svm_hpo = time.time() - start_svm

print(f"Mejor LR: {grid_lr.best_params_} (Tiempo: {time_lr_hpo:.2f}s)")
print(f"Mejor SVM: {grid_svm.best_params_} (Tiempo: {time_svm_hpo:.2f}s)")

#### **Extracción de atributos relevantes**

El enunciado pregunta si es posible extraer qué atributos son más importantes. En los modelos lineales (regresión logística), esto se hace analizando los coeficientes (*coef_*).

In [ ]:
# 1. Extraemos los coeficientes de la Regresión Logística
best_lr = grid_lr.best_estimator_.named_steps['clf']
feature_names = grid_lr.best_estimator_.named_steps['preprocessor'].get_feature_names_out()

# AQUÍ ESTÁ EL ARREGLO: best_lr.coef_[0]
coefficients = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': best_lr.coef_[0] 
}).sort_values(by='Coefficient', ascending=False)

print("--- ATRIBUTOS MÁS RELEVANTES (Regresión Logística) ---")
print(coefficients.head(10)) # Variables que más influyen positivamente en el 'Sí'
print("\n" + "-"*50 + "\n")

# 2. Evaluación final de los mejores modelos en los datos de Test
y_pred_lr = grid_lr.predict(X_test)
y_pred_svm = grid_svm.predict(X_test)

print("--- REPORTE DE CLASIFICACIÓN (Mejor Regresión Logística) ---")
print(classification_report(y_test, y_pred_lr))

print("\n--- REPORTE DE CLASIFICACIÓN (Mejor SVM) ---")
print(classification_report(y_test, y_pred_svm))

#### **Justificación gráfica de los modelos avanzados**

Para respaldar las métricas obtenidas, hemos generado dos representaciones visuales clave:

**1. Importancia de las variables (regresión logística):**
El gráfico de barras ilustra el peso de los coeficientes del modelo lineal. Visualmente comprobamos que atributos como tener un éxito previo en otra campaña (*cat__poutcome_success*) o la duración de la llamada (*num__duration*) son los motores principales que impulsan la probabilidad de contratación. Esto hace que el modelo sea altamente interpretable para el equipo de negocio.

**2. Comparativa de matrices de confusión:**
Al observar los mapas de calor, el rendimiento superior del **SVM** queda perfectamente reflejado en los números. 
Si nos fijamos en los clientes que *Realmente* querían el depósito (la fila inferior de ambas matrices):
* La **regresión logística** logró captar a 1.342 clientes, pero dejó escapar a 398 (Falsos Negativos).
* El **SVM**, sin embargo, logró identificar correctamente a **1.530 clientes**, reduciendo las fugas a solo 210. 

A nivel de negocio, esto significa que el SVM es capaz de encontrar a **más clientes rentables** que la regresión logística en este conjunto de prueba, asumiendo un coste operativo ínfimo. La rentabilidad y superioridad del SVM es indiscutible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("--- GENERANDO GRÁFICOS VISUALES PARA MODELOS AVANZADOS ---")

# 1. Gráfico de Importancia de Variables (Solo para Regresión Logística)
plt.figure(figsize=(10, 6))
# Usamos el dataframe 'coefficients' que creamos en el paso anterior
sns.barplot(x='Coefficient', y='Feature', data=coefficients.head(10), palette='viridis')
plt.title('Top 10 Variables más determinantes para predecir "Sí" (Reg. Logística)', fontsize=14)
plt.xlabel('Peso (Coeficiente)', fontsize=12)
plt.ylabel('Atributo del Cliente', fontsize=12)
plt.tight_layout()
plt.show()

# 2. Matrices de Confusión (Comparativa visual LR vs SVM)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz LR
sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predice NO', 'Predice YES'], 
            yticklabels=['Real NO', 'Real YES'], ax=axes[0])
axes[0].set_title('Matriz de Confusión - Regresión Logística')

# Matriz SVM
sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, fmt='d', cmap='Purples', 
            xticklabels=['Predice NO', 'Predice YES'], 
            yticklabels=['Real NO', 'Real YES'], ax=axes[1])
axes[1].set_title('Matriz de Confusión - SVM (Ganador)')

plt.tight_layout()
plt.show()

#### **Análisis de resultados de los modelos avanzados**

Tras evaluar la regresión logística y las SVMs mediante búsqueda de hiperparámetros, sacamos las siguientes conclusiones:

##### Regresión logística
* **Configuración óptima:** El modelo escogió una regularización L2 (`penalty='l2'`) con un valor de `C=10`.
* **Métricas:** Logra una Exactitud del **83%**. Su Precisión para la clase 'yes' es altísima (0.85), pero su Recall baja a 0.77. Es un modelo que se equivoca muy poco cuando dice "Sí", pero deja escapar bastantes clientes.
* **Importancia de las variables (coeficientes):** La ventaja de este modelo es que es totalmente explicable. Al extraer sus coeficientes, comprobamos que variables como la **duración de la llamada** (*num__duration*), el éxito en campañas previas (*cat__poutcome_success*), y contactar en **Marzo o Diciembre** influyen de manera muy positiva en la contratación del depósito.

##### Support Vector Machines (SVM)
* **Configuración óptima:** El mejor modelo utiliza un kernel radial (`kernel='rbf'`), `C=1` y `gamma='scale'`.
* **Métricas:** Los resultados son excepcionales. Alcanza una **exactitud (accuracy) del 86%**. Al observar los promedios globales, vemos que logra un equilibrio perfecto (F1-Score de 0.86), disparando tanto la precisión como el recall muy por encima de los otros modelos.

## **5. Conclusiones y selección del mejor modelo**

A lo largo de esta práctica hemos evaluado cuatro algoritmos de distinta naturaleza: KNN, Árboles de Decisión, Regresión Logística y SVM.

El **ganador indiscutible para nuestro problema bancario son las SVMs (Support Vector Machines)**. 

* **Comparativa:** Mientras que nuestro mejor modelo básico (el Árbol de Decisión) lograba una exactitud del 82%, un Recall del 79% y una precisión del 83%, el **SVM eleva la exactitud al 86%**, mejorando de forma drástica tanto la capacidad de encontrar a los clientes reales (*Recall*) como la fiabilidad de sus predicciones (*Precisión*).
* **Impacto de negocio:** Utilizando el modelo SVM, el banco optimizará al máximo su campaña telefónica. Minimizará el coste de llamadas a clientes no interesados y, lo más importante, maximizará la captación de aquellos que sí lo están, logrando la mayor rentabilidad posible de todos los modelos probados.

Este modelo SVM será el que reentrenaremos con el 100% de los datos para realizar la predicción ciega de la competición y el que desplegaremos en producción.